# OpenKind 4E-B.1: candidate-conditioned option-logit audit

**Corrected prepared experiment, not an executed result. Final split remains closed.**

## Question

Can the pinned Qwen3.5-4B-Base model rank the existing candidates from one constrained answer-token readout, and can an explicit `Z` token identify semantic none? This follows the JevK5 style of listing answer options in the prompt and reading the next answer token. It tests that method on OpenKind's frozen model and corpus; it does not import JevK5's checkpoint, reported scores, or task distribution.

The previous 4E-B.0 root-only and cheap-probe arms failed their nonfinal gates. This audit adds full candidate-conditioned transformer execution after state prefill. QASPER has one explicit `answerable` candidate plus semantic none, so its useful diagnostic is `A` versus `Z`; it has no multi-candidate ranking task. This is a fixed diagnostic, with no training, prompt sweep, threshold fit, final evaluation, or promotion claim.

Sources: [JevK5](https://github.com/allebee/jevk5), [OpenKind research](https://github.com/whit3rabbit/openpick/blob/main/docs/RESEARCH.md). The linked repository is methodological context, not a performance baseline.


## Fixed protocol

- Model: `Qwen/Qwen3.5-4B-Base`, revision `1001bb4d826a52d1f399e183466143f4da7b741b`, FP32 text graph, no TF32, no generation.
- Input: exact Phase 4A effective state text, then question, ordered candidate descriptions, and an explicit `Z. None of the listed options is supported by the context.`
- Readout: one next-token logit for each candidate letter and `Z`; softmax of those logits. No temperature or thresholds are fitted.
- Sample: first 16 calibration-gate states per source by SHA-256 of state ID, all choice questions in those states. Selection does not inspect labels. Seed 17; ContractNLI and QASPER only.
- Primary comparison: answerable-only conditional accuracy and NLL on questions with at least two listed candidates, versus the frozen Phase 4A reference on identical question IDs. QASPER's one-option `A` versus `Z` answerability score is reported separately as a diagnostic.
- Order diagnostic: original option order and reversed order for multi-candidate questions, with option IDs mapped back before comparison. Report disagreement and probability shifts. Original order is primary.
- Cache guard: verify a deep-copied hybrid state cache does not alias the root and that cached and full-prompt logits agree on the first question from each source. Stop on mismatch.
- Timing: measured only for this exact Colab host and sample, split into root prefill and question suffix. Model download/load is excluded. No throughput or service claim.

Run a GPU runtime with at least the FP32 memory guard below. The notebook writes an immutable contract before model execution and a result lock only after completion. Existing result directories are never overwritten. Outputs are exploratory and cannot change the frozen 4E-A disposition or open final data.


In [1]:
# Install the known parser/runtime version without replacing Colab's Torch build.
import importlib.metadata as im
import subprocess, sys

TORCH_VERSION = im.version('torch')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    f'torch=={TORCH_VERSION}', 'transformers==5.17.0',
    'numpy>=2,<3', 'pandas>=2,<3', 'pyarrow>=18,<30',
])

import copy, hashlib, json, math, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

drive.mount('/content/drive')
assert torch.__version__.startswith('2.11.'), f'Expected Torch 2.11.x, found {torch.__version__}'
assert torch.cuda.is_available(), 'Select a Colab GPU runtime; no CPU fallback for this model.'
torch.set_float32_matmul_precision('highest')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
print('Torch:', torch.__version__, 'Transformers:', im.version('transformers'))
print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
Torch: 2.11.0+cu128 Transformers: 5.17.0
GPU: NVIDIA L4


In [2]:
SEED = 17
MAX_STATES_PER_SOURCE = 16
RUN_LABEL = 'candidate_option_logit_gate16_s17_v2'
MODEL_ID = 'Qwen/Qwen3.5-4B-Base'
MODEL_REVISION = '1001bb4d826a52d1f399e183466143f4da7b741b'
PROFILE_ID = 'a047d6802c3f06f085b8'
BUNDLE_SHA256 = '4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332'
BASE_RUN_ID = '20260921T013558Z'
SOURCES = ('contractnli', 'qasper')
SPLIT = 'calibration_gate'
MAX_FULL_TOKENS = 4096
PARITY_LOGIT_TOLERANCE = 0.05
PARITY_PROB_TOLERANCE = 0.01

COLAB_ROOT = Path('/content/drive/MyDrive/Colab Notebooks')
BASE_RUN = COLAB_ROOT / 'OpenDecision_Phase4A_StateQuery_results' / BASE_RUN_ID
CORPUS_ROOT = BASE_RUN / 'opendecision-mq-v1'
FEATURE_ROOT = BASE_RUN / 'state_features'
PHASE4D_RUN = (COLAB_ROOT / 'OpenDecision_Phase4D_EvidenceAware_Applicability_results'
               / BASE_RUN_ID / 'b2_evidence_residual_s17')
RESULT_DIR = COLAB_ROOT / 'OpenKind_Phase4E_B_Candidate_Option_Logit_results' / RUN_LABEL
assert not RESULT_DIR.exists(), f'Immutable run already exists: {RESULT_DIR}'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def json_read(path):
    return json.loads(Path(path).read_text())

def canonical_sha(value):
    encoded = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False,
                         allow_nan=False).encode()
    return hashlib.sha256(encoded).hexdigest()

def write_once(path, value):
    path = Path(path)
    if path.exists():
        raise FileExistsError(path)
    temp = path.with_name(path.name + '.partial')
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False))
    os.replace(temp, path)

required = [BASE_RUN / 'DECLARED_CONTRACT.json', BASE_RUN / 'MODEL_SELECTION_LOCK.json',
            BASE_RUN / 'NONFINAL_EVALUATION.json', BASE_RUN / 'reference_nonfinal_predictions.parquet',
            FEATURE_ROOT / 'manifest.json', CORPUS_ROOT / 'manifest.json',
            CORPUS_ROOT / 'corpus_lock.json', PHASE4D_RUN / 'NONFINAL_EVALUATION.json',
            PHASE4D_RUN / 'NONFINAL_RESULT_LOCK.json']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(missing)
if (BASE_RUN / 'FINAL_EVALUATION.json').exists() or (PHASE4D_RUN / 'FINAL_EVALUATION.json').exists():
    raise RuntimeError('A final result is present. Stop and re-audit the evidence boundary.')

declared = json_read(BASE_RUN / 'DECLARED_CONTRACT.json')
model_record = declared['model']
assert (model_record['model_id'], model_record['revision'], model_record['profile_id'],
        model_record['bundle_sha256']) == (MODEL_ID, MODEL_REVISION, PROFILE_ID, BUNDLE_SHA256)
assert declared['declared_before_data_or_predictions'] is True
corpus_manifest = json_read(CORPUS_ROOT / 'manifest.json')
corpus_lock = json_read(CORPUS_ROOT / 'corpus_lock.json')
assert corpus_manifest['run_id'] == BASE_RUN_ID
assert corpus_lock['manifest_sha256'] == canonical_sha(corpus_manifest)
assert corpus_manifest.get('final_labels_or_predictions_opened') is False
for table, expected in corpus_lock['tables'].items():
    actual = sha256_file(CORPUS_ROOT / f'{table}.parquet')
    if actual != expected:
        raise RuntimeError(f'Corpus table changed: {table}')

features = json_read(FEATURE_ROOT / 'manifest.json')
assert features['run_id'] == BASE_RUN_ID
assert features['corpus_manifest_sha256'] == corpus_lock['manifest_sha256']
assert features['reference_predictions_exclude_final'] is True
phase4d = json_read(PHASE4D_RUN / 'NONFINAL_EVALUATION.json')
phase4d_lock = json_read(PHASE4D_RUN / 'NONFINAL_RESULT_LOCK.json')
assert phase4d.get('final_opened') is False and phase4d_lock.get('final_opened') is False
assert phase4d_lock['nonfinal_report_sha256'] == sha256_file(PHASE4D_RUN / 'NONFINAL_EVALUATION.json')

# Filters keep final corpus rows out of the data frames. The feature manifest
# contains metadata for all states, but only selected nonfinal text is dereferenced.
states = pd.read_parquet(CORPUS_ROOT / 'states.parquet',
                         filters=[('split', '==', SPLIT)], columns=['state_id', 'source', 'split'])
states = states.loc[states['source'].isin(SOURCES)].copy()
states['sample_key'] = states['state_id'].map(lambda s: hashlib.sha256(str(s).encode()).hexdigest())
selected_states = (states.sort_values(['source', 'sample_key', 'state_id'])
                   .groupby('source', sort=True).head(MAX_STATES_PER_SOURCE).copy())
assert set(selected_states['source']) == set(SOURCES)
assert selected_states.groupby('source').size().eq(MAX_STATES_PER_SOURCE).all()
selected_state_ids = set(selected_states['state_id'])

questions = pd.read_parquet(CORPUS_ROOT / 'questions.parquet',
                            filters=[('state_id', 'in', sorted(selected_state_ids))])
questions = questions.loc[(questions['state_id'].isin(selected_state_ids))
                          & (questions['primitive'] == 'choice')].copy()
assert len(questions) > 0 and questions['question_id'].is_unique
if 'split' in questions:
    assert questions['split'].eq(SPLIT).all()
options = pd.read_parquet(CORPUS_ROOT / 'options.parquet',
                          filters=[('question_id', 'in', sorted(questions['question_id']))])
assert set(options['question_id']) == set(questions['question_id'])
reference = pd.read_parquet(BASE_RUN / 'reference_nonfinal_predictions.parquet',
                            filters=[('question_id', 'in', sorted(questions['question_id']))])
assert reference['question_id'].is_unique
assert set(reference['question_id']) == set(questions['question_id'])

state_feature_index = {row['state_id']: row for row in features['state_files']}
assert selected_state_ids <= set(state_feature_index)
option_groups = {qid: group.sort_values('position').to_dict('records')
                 for qid, group in options.groupby('question_id')}
reference_index = reference.set_index('question_id').to_dict('index')
questions = questions.merge(selected_states[['state_id', 'source']], on='state_id',
                            how='left', validate='many_to_one', suffixes=('', '_state'))
if 'source_state' in questions:
    assert questions['source'].eq(questions['source_state']).all()
    questions = questions.drop(columns='source_state')
assert questions['source'].isin(SOURCES).all()

contract = {
    'schema': 'openkind-candidate-option-logit-audit/v2', 'seed': SEED,
    'run_label': RUN_LABEL, 'split': SPLIT, 'sources': list(SOURCES),
    'max_states_per_source': MAX_STATES_PER_SOURCE,
    'selected_state_ids': selected_states.sort_values(['source', 'sample_key'])['state_id'].tolist(),
    'selected_question_ids_sha256': canonical_sha(sorted(questions['question_id'].tolist())),
    'question_count': len(questions), 'model': model_record,
    'input_sha256': {str(path.relative_to(COLAB_ROOT)): sha256_file(path)
                     for path in required},
    'parity_logit_tolerance': PARITY_LOGIT_TOLERANCE,
    'parity_probability_tolerance': PARITY_PROB_TOLERANCE,
    'primary_order': 'original', 'diagnostic_order': 'reversed',
    'none_action': 'Z', 'final_opened': False, 'training_performed': False,
}
RESULT_DIR.mkdir(parents=True, exist_ok=False)
write_once(RESULT_DIR / 'EXPERIMENT_CONTRACT.json', contract)
rows, parity_rows, verified_sources = [], [], set()
print('Frozen', len(selected_states), 'states and', len(questions), 'questions.')
print('Contract:', RESULT_DIR / 'EXPERIMENT_CONTRACT.json')


Frozen 32 states and 325 questions.
Contract: /content/drive/MyDrive/Colab Notebooks/OpenKind_Phase4E_B_Candidate_Option_Logit_results/candidate_option_logit_gate16_s17_v2/EXPERIMENT_CONTRACT.json


## Load the pinned FP32 text model

The text model and vocabulary head are kept separate so the readout projects only the final hidden token. The notebook stops if the model class, parameter count, precision, or available memory differs from the frozen reference. It never silently uses BF16 or a smaller model.


In [3]:
free_bytes, total_bytes = torch.cuda.mem_get_info()
if free_bytes / 2**30 < 19.0:
    raise RuntimeError(f'Need at least 19 GiB free GPU memory; found {free_bytes / 2**30:.1f} GiB.')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
                                           trust_remote_code=False)
model, load_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False,
    dtype=torch.float32, attn_implementation='sdpa', device_map={'': 0},
    output_loading_info=True)
assert type(model).__name__ == 'Qwen3_5ForCausalLM'
assert type(model.model).__name__ == 'Qwen3_5TextModel'
assert sum(p.numel() for p in model.model.parameters()) == 4205751296
assert not any(load_info.get(k) for k in ('missing_keys', 'mismatched_keys', 'error_msgs'))
assert next(model.parameters()).dtype == torch.float32
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
torch.manual_seed(SEED)
print('Model loaded. Free GPU GiB:', round(torch.cuda.mem_get_info()[0] / 2**30, 2))


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Model loaded. Free GPU GiB: 6.13


In [4]:
LETTERS = 'ABCDEFGHIJKLMNOPQRSTUVWXY'  # Z is reserved for semantic none.

def sync():
    torch.cuda.synchronize()

def action_token(letter):
    # The action is one constrained next token, without generation.
    ids = tokenizer.encode(' ' + letter, add_special_tokens=False)
    if len(ids) != 1 or tokenizer.decode(ids) != ' ' + letter:
        raise RuntimeError(f'Answer action is not a single exact token: {letter!r} -> {ids}')
    return ids[0]

action_ids = {letter: action_token(letter) for letter in LETTERS + 'Z'}
assert len(set(action_ids.values())) == len(action_ids)

def prefix_text(state_id):
    effective = state_feature_index[state_id]['effective_state_text']
    return 'Context:\n' + effective + '\n\n'

def suffix_text(question, ordered_options):
    if not 1 <= len(ordered_options) <= len(LETTERS):
        raise RuntimeError('Unsupported candidate count.')
    lines = [
        'Question: ' + str(question['instruction']),
        'Choose the single best option supported by the context. Choose Z if none is supported.',
    ]
    lines.extend(f'{LETTERS[i]}. {option["description"]}'
                 for i, option in enumerate(ordered_options))
    lines.extend(['Z. None of the listed options is supported by the context.', 'Answer:'])
    return '\n'.join(lines)

def encode_text(text):
    return tokenizer.encode(text, add_special_tokens=False)

def model_inputs(ids, prefix_len=0):
    return {
        'input_ids': torch.tensor([ids], dtype=torch.long, device='cuda:0'),
        'attention_mask': torch.ones((1, prefix_len + len(ids)), dtype=torch.long, device='cuda:0'),
        'position_ids': torch.arange(prefix_len, prefix_len + len(ids), device='cuda:0')[None, :],
    }

def cache_tensors(obj, seen=None):
    seen = set() if seen is None else seen
    if id(obj) in seen:
        return
    seen.add(id(obj))
    if isinstance(obj, torch.Tensor):
        yield obj
    elif isinstance(obj, dict):
        for value in obj.values():
            yield from cache_tensors(value, seen)
    elif isinstance(obj, (list, tuple)):
        for value in obj:
            yield from cache_tensors(value, seen)
    elif hasattr(obj, '__dict__'):
        for key, value in vars(obj).items():
            if key != 'prefetch_stream':
                yield from cache_tensors(value, seen)

def cache_signature(cache):
    h = hashlib.sha256()
    for tensor in cache_tensors(cache):
        h.update(str((tuple(tensor.shape), str(tensor.dtype))).encode())
        h.update(tensor.detach().contiguous().view(torch.uint8).cpu().numpy().tobytes())
    return h.hexdigest()

def cache_storages(cache):
    return {t.untyped_storage().data_ptr() for t in cache_tensors(cache) if t.numel()}

def project_last(hidden, letters):
    raw = model.lm_head(hidden[:, -1:, :]).float()[0, 0]
    return np.asarray([float(raw[action_ids[c]].item()) for c in letters], dtype=np.float64)

def prefill(ids):
    with torch.inference_mode():
        out = model.model(**model_inputs(ids), use_cache=True, return_dict=True)
    cache = out.past_key_values
    assert cache is not None and int(cache.get_seq_length()) == len(ids)
    return cache

def cached_logits(root_cache, root_len, suffix_ids, letters, verify_copy=False):
    before = cache_signature(root_cache) if verify_copy else None
    branch = copy.deepcopy(root_cache)
    if verify_copy:
        assert not (cache_storages(root_cache) & cache_storages(branch)), 'Hybrid cache aliases root.'
        assert cache_signature(branch) == before
    with torch.inference_mode():
        out = model.model(**model_inputs(suffix_ids, root_len), past_key_values=branch,
                          use_cache=True, return_dict=True)
        logits = project_last(out.last_hidden_state, letters)
    assert int(out.past_key_values.get_seq_length()) == root_len + len(suffix_ids)
    if verify_copy:
        assert cache_signature(root_cache) == before, 'Root cache mutated by branch.'
    return logits

def full_logits(root_ids, suffix_ids, letters):
    with torch.inference_mode():
        out = model.model(**model_inputs(root_ids + suffix_ids), use_cache=False, return_dict=True)
        return project_last(out.last_hidden_state, letters)

def softmax(logits):
    shifted = np.asarray(logits, dtype=np.float64) - np.max(logits)
    weights = np.exp(shifted)
    return weights / weights.sum()

def score_order(question, ordered_options, root_ids, root_cache, verify_copy=False):
    suffix = suffix_text(question, ordered_options)
    suffix_ids = encode_text(suffix)
    if len(root_ids) + len(suffix_ids) > MAX_FULL_TOKENS:
        raise RuntimeError(f'Prompt exceeds fixed {MAX_FULL_TOKENS} token bound.')
    letters = LETTERS[:len(ordered_options)] + 'Z'
    sync(); started = time.perf_counter()
    logits = cached_logits(root_cache, len(root_ids), suffix_ids, letters, verify_copy)
    sync(); elapsed = time.perf_counter() - started
    probs = softmax(logits)
    mapping = {option['option_id']: float(probs[i]) for i, option in enumerate(ordered_options)}
    return {'option_probabilities': mapping, 'none_probability': float(probs[-1]),
            'logits': [float(v) for v in logits], 'suffix_tokens': len(suffix_ids),
            'suffix_seconds': elapsed, 'suffix_ids': suffix_ids, 'letters': letters,
            'suffix_token_sha256': canonical_sha(suffix_ids)}


## Run the bounded audit

The loop groups questions by state so the state prefix is computed once. The first question of each source checks cache versus full-prompt outputs. Multi-candidate questions are evaluated in original and reversed option order. A parity failure stops the run before summary or result lock. If a recoverable error occurs, rerunning this cell in the same kernel skips completed question IDs. Do not rerun the setup cell that created the immutable contract.


In [5]:
# Preserve completed in-memory rows on a same-kernel retry.
rows = globals().get('rows', [])
parity_rows = globals().get('parity_rows', [])
verified_sources = globals().get('verified_sources', set())
completed_question_ids = {row['question_id'] for row in rows}
assert len(completed_question_ids) == len(rows), 'Duplicate question in resumed rows.'
question_by_state = {sid: group.sort_values('question_id').to_dict('records')
                     for sid, group in questions.groupby('state_id')}
ordered_states = selected_states.sort_values(['source', 'sample_key', 'state_id']).to_dict('records')

for state_number, state in enumerate(ordered_states, 1):
    state_id = state['state_id']
    pending = [q for q in question_by_state.get(state_id, [])
               if q['question_id'] not in completed_question_ids]
    if not pending:
        continue
    root_ids = encode_text(prefix_text(state_id))
    sync(); started = time.perf_counter()
    root_cache = prefill(root_ids)
    sync(); root_seconds = time.perf_counter() - started
    for question in pending:
        qid = question['question_id']
        original_options = option_groups[qid]
        verify = state['source'] not in verified_sources
        original = score_order(question, original_options, root_ids, root_cache, verify_copy=verify)
        if verify:
            expected = full_logits(root_ids, original['suffix_ids'], original['letters'])
            full_prob = softmax(expected)
            cached_prob = softmax(original['logits'])
            max_logit_diff = float(np.max(np.abs(expected - original['logits'])))
            max_prob_diff = float(np.max(np.abs(full_prob - cached_prob)))
            same_action = int(np.argmax(expected)) == int(np.argmax(original['logits']))
            parity_rows.append({'source': state['source'], 'question_id': qid,
                                'max_abs_logit_diff': max_logit_diff,
                                'max_abs_probability_diff': max_prob_diff,
                                'same_action': same_action})
            if (max_logit_diff > PARITY_LOGIT_TOLERANCE
                    or max_prob_diff > PARITY_PROB_TOLERANCE or not same_action):
                raise RuntimeError(f'Cached/full-prompt mismatch on {qid}: {parity_rows[-1]}')
            verified_sources.add(state['source'])
        reversed_order = (score_order(question, list(reversed(original_options)), root_ids,
                                      root_cache) if len(original_options) > 1 else None)
        target_kind = question['target_kind']
        target_option_id = None if target_kind == 'semantic_none' else question['target_option_id']
        if target_option_id is not None:
            assert target_option_id in original['option_probabilities']
        ref = reference_index[qid]
        ref_options = json.loads(ref['option_probabilities_json'])
        assert set(ref_options) == set(original['option_probabilities'])
        rows.append({
            'source': state['source'], 'state_id': state_id, 'question_id': qid,
            'split': SPLIT, 'target_kind': target_kind, 'target_option_id': target_option_id,
            'option_ids_json': json.dumps([o['option_id'] for o in original_options]),
            'primary_option_probabilities_json': json.dumps(original['option_probabilities'], sort_keys=True),
            'reverse_option_probabilities_json': (json.dumps(reversed_order['option_probabilities'], sort_keys=True)
                                                  if reversed_order else None),
            'primary_none_probability': original['none_probability'],
            'reverse_none_probability': reversed_order['none_probability'] if reversed_order else None,
            'reference_option_probabilities_json': json.dumps(ref_options, sort_keys=True),
            'reference_none_probability': float(ref['none_probability']),
            'root_tokens': len(root_ids), 'primary_suffix_tokens': original['suffix_tokens'],
            'reverse_suffix_tokens': reversed_order['suffix_tokens'] if reversed_order else None,
            'root_token_sha256': canonical_sha(root_ids),
            'primary_suffix_token_sha256': original['suffix_token_sha256'],
            'reverse_suffix_token_sha256': reversed_order['suffix_token_sha256'] if reversed_order else None,
            'primary_logits_json': json.dumps(original['logits']),
            'reverse_logits_json': json.dumps(reversed_order['logits']) if reversed_order else None,
            'root_seconds_amortized': root_seconds / len(pending),
            'primary_suffix_seconds': original['suffix_seconds'],
            'reverse_suffix_seconds': reversed_order['suffix_seconds'] if reversed_order else None,
        })
        completed_question_ids.add(qid)
    del root_cache
    torch.cuda.empty_cache()
    print(f'{state_number}/{len(ordered_states)} states, {len(rows)} questions', flush=True)

assert verified_sources == set(SOURCES)
assert len(rows) == len(questions)
assert set(row['question_id'] for row in rows) == set(questions['question_id'])
print('Completed', len(rows), 'nonfinal questions. Cache parity:', parity_rows)


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


1/32 states, 17 questions
2/32 states, 34 questions
3/32 states, 51 questions
4/32 states, 68 questions
5/32 states, 85 questions
6/32 states, 102 questions
7/32 states, 119 questions
8/32 states, 136 questions
9/32 states, 153 questions
10/32 states, 170 questions
11/32 states, 187 questions
12/32 states, 204 questions
13/32 states, 221 questions
14/32 states, 238 questions
15/32 states, 255 questions
16/32 states, 272 questions
17/32 states, 277 questions
18/32 states, 278 questions
19/32 states, 282 questions
20/32 states, 285 questions
21/32 states, 288 questions
22/32 states, 289 questions
23/32 states, 294 questions
24/32 states, 298 questions
25/32 states, 300 questions
26/32 states, 305 questions
27/32 states, 308 questions
28/32 states, 310 questions
29/32 states, 312 questions
30/32 states, 316 questions
31/32 states, 321 questions
32/32 states, 325 questions
Completed 325 nonfinal questions. Cache parity: [{'source': 'contractnli', 'question_id': 'contractnli:test:465:nda-1'

In [6]:
def argmax_id(probabilities):
    return max(probabilities, key=lambda key: probabilities[key])

def conditional_metrics(group, field, none_field):
    scored = []
    for row in group:
        if row['target_option_id'] is None or len(json.loads(row['option_ids_json'])) < 2:
            continue
        probabilities = json.loads(row[field])
        remaining = sum(probabilities.values())
        if remaining <= 0:
            continue
        conditional = {key: value / remaining for key, value in probabilities.items()}
        scored.append((row, conditional))
    if not scored:
        return {'n': 0, 'accuracy': None, 'nll': None}
    return {
        'n': len(scored),
        'accuracy': sum(argmax_id(p) == row['target_option_id'] for row, p in scored) / len(scored),
        'nll': sum(-math.log(max(p[row['target_option_id']], 1e-12))
                   for row, p in scored) / len(scored),
    }

def full_metrics(group, field, none_field):
    if not group:
        return {'n': 0}
    correct = 0
    nll = brier = none_true = none_hit = none_false = answerable = 0
    for row in group:
        probabilities = json.loads(row[field])
        probabilities['__NONE__'] = row[none_field]
        total = sum(probabilities.values())
        if not 0.999 <= total <= 1.001:
            raise RuntimeError(f'Probability mass changed on {row["question_id"]}: {total}')
        target = row['target_option_id'] or '__NONE__'
        pred = argmax_id(probabilities)
        correct += pred == target
        nll += -math.log(max(probabilities[target], 1e-12))
        brier += sum((value - (key == target))**2 for key, value in probabilities.items())
        if target == '__NONE__':
            none_true += 1
            none_hit += pred == target
        else:
            answerable += 1
            none_false += pred == '__NONE__'
    return {'n': len(group), 'accuracy': correct / len(group), 'nll': nll / len(group),
            'brier': brier / len(group), 'semantic_none_n': none_true,
            'semantic_none_recall': none_hit / none_true if none_true else None,
            'answerable_n': answerable,
            'false_none_rate': none_false / answerable if answerable else None}

summary = {'schema': 'openkind-candidate-option-logit-nonfinal/v2',
           'run_label': RUN_LABEL, 'sample_n': len(rows), 'cache_parity': parity_rows,
           'by_source': {}, 'limits': [
               'Fixed 16-state-per-source exploratory sample, not the full calibration gate.',
               'No prompt or threshold selection, training, final split, or independent review.',
               'Reference and option-logit prompts differ; this is a method comparison on matched rows.',
               'FP32 Colab timing is host-specific and excludes model load and service overhead.',
               'Reference PyTorch causal convolution and DeltaNet fallbacks may make this host slower.',
           ]}
for source in SOURCES:
    group = [row for row in rows if row['source'] == source]
    primary = conditional_metrics(group, 'primary_option_probabilities_json',
                                  'primary_none_probability')
    reference_conditional = conditional_metrics(group, 'reference_option_probabilities_json',
                                                'reference_none_probability')
    full = full_metrics(group, 'primary_option_probabilities_json',
                        'primary_none_probability')
    reference_full = full_metrics(group, 'reference_option_probabilities_json',
                                  'reference_none_probability')
    multi_candidate = [row for row in group if len(json.loads(row['option_ids_json'])) > 1]
    candidate_counts = {str(k): sum(len(json.loads(row['option_ids_json'])) == k for row in group)
                        for k in sorted({len(json.loads(row['option_ids_json'])) for row in group})}
    order_disagreement = sum(
        argmax_id(json.loads(row['primary_option_probabilities_json'])) !=
        argmax_id(json.loads(row['reverse_option_probabilities_json']))
        for row in multi_candidate)
    order_max_shift = max((
        abs(json.loads(row['primary_option_probabilities_json'])[oid] -
            json.loads(row['reverse_option_probabilities_json'])[oid])
        for row in multi_candidate for oid in json.loads(row['option_ids_json'])), default=None)
    summary['by_source'][source] = {
        'candidate_count_distribution': candidate_counts,
        'conditional_option_logit': primary,
        'conditional_frozen_reference': reference_conditional,
        'full_option_logit_diagnostic': full,
        'full_frozen_reference': reference_full,
        'reverse_order_multi_candidate_n': len(multi_candidate),
        'reverse_order_candidate_winner_disagreement_n': order_disagreement,
        'reverse_order_max_option_probability_shift': order_max_shift,
        'mean_root_seconds_amortized': float(np.mean([r['root_seconds_amortized'] for r in group])),
        'mean_primary_suffix_seconds': float(np.mean([r['primary_suffix_seconds'] for r in group])),
        'mean_reverse_suffix_seconds': (float(np.mean([r['reverse_suffix_seconds']
                                                    for r in multi_candidate]))
                                        if multi_candidate else None),
    }
print(json.dumps(summary, indent=2, allow_nan=False))


{
  "schema": "openkind-candidate-option-logit-nonfinal/v2",
  "run_label": "candidate_option_logit_gate16_s17_v2",
  "sample_n": 325,
  "cache_parity": [
    {
      "source": "contractnli",
      "question_id": "contractnli:test:465:nda-1",
      "max_abs_logit_diff": 7.62939453125e-06,
      "max_abs_probability_diff": 2.5505493301269944e-06,
      "same_action": true
    },
    {
      "source": "qasper",
      "question_id": "qasper:test:1708.05482:67c16ba64fe27838b1034d15194c07a9c98cdebe",
      "max_abs_logit_diff": 1.9073486328125e-05,
      "max_abs_probability_diff": 3.07335971005962e-07,
      "same_action": true
    }
  ],
  "by_source": {
    "contractnli": {
      "candidate_count_distribution": {
        "2": 272
      },
      "conditional_option_logit": {
        "n": 148,
        "accuracy": 0.8783783783783784,
        "nll": 0.4217188433706784
      },
      "conditional_frozen_reference": {
        "n": 148,
        "accuracy": 0.3108108108108108,
        "nll": 1.6

In [7]:
# Save only after the complete run. The result lock binds every output to its contract.
rows_path = RESULT_DIR / 'NONFINAL_ROWS.parquet'
summary_path = RESULT_DIR / 'NONFINAL_EVALUATION.json'
lock_path = RESULT_DIR / 'NONFINAL_RESULT_LOCK.json'
if any(p.exists() for p in (rows_path, summary_path, lock_path)):
    raise FileExistsError('Result file already exists. Use a fresh run label.')
pd.DataFrame(rows).to_parquet(rows_path, index=False)
write_once(summary_path, summary)
write_once(lock_path, {
    'schema': 'openkind-candidate-option-logit-result-lock/v1',
    'run_label': RUN_LABEL,
    'contract_sha256': sha256_file(RESULT_DIR / 'EXPERIMENT_CONTRACT.json'),
    'rows_sha256': sha256_file(rows_path),
    'nonfinal_report_sha256': sha256_file(summary_path),
    'row_count': len(rows), 'final_opened': False,
})
print('Saved:', RESULT_DIR)


Saved: /content/drive/MyDrive/Colab Notebooks/OpenKind_Phase4E_B_Candidate_Option_Logit_results/candidate_option_logit_gate16_s17_v2


## Reading the result

The conditional comparison answers whether answer-token logits can rank listed candidates on answerable questions with at least two candidates. QASPER's one-candidate rows contribute to the full `A` versus `Z` answerability diagnostic, never to the ranking comparison or order test. Inspect both source rows and cache parity. A competitive conditional score with weak `Z` behavior supports a narrower candidate-ranking use, not replacement of the complete decision head. Any material order instability or cache mismatch limits the method. The frozen final split stays closed; a larger or confirmatory experiment requires a new contract and independent review.
